# TGLF SAT1 (EM width search) vs SAT2 (ES width search) vs GS2 on the four `kbm_free` KBM databases (Fusion_PhD-qp5.35)

Advisory numbers only. Same populations, species ties and bias statistic as `tglf_filter_width.ipynb` (qp5.34); adds the SAT2 arms (`s2_*` leaves) and the $\theta$-extent of $\phi$.

- SAT1 = the qp5.34 `fw_*` leaves (EM search: fields on in the width search). SAT2 = `s2_*` (ES search, since `WIDTH_MIN` $>0$). The SAT2 presets also switch `XNU_MODEL` 2$\to$3 and `WDIA_TRAPPED` 0$\to$1 (`tglf_startup.f90`), so SAT1 vs SAT2 is two TGLF configurations, not the search alone.
- Extent: pyro `Extent` (95% of $\int|\phi|^2\,d\theta$). TGLF: the leaf's `phi`, mode 0 (dominant, checked). GS2: the cube's `phi_squared`.
- Ratio $\theta_{\rm TGLF}/\theta_{\rm GS2}$ is the TGLF extent over the GS2 extent for the same sample.

In [ ]:
import os
for threads in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[threads] = "1"       # one BLAS thread per worker process (qp5.33 oversubscription)
import json, re, tempfile, warnings
from multiprocessing import get_context
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.stats import spearmanr
import pyrokinetics
from pyrokinetics import Pyro, PyroHypercube
from pyrokinetics.diagnostics.extent import Extent
from general_analysis import mode_classification as mc

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)
warnings.filterwarnings("ignore")
print("pyrokinetics", pyrokinetics.__version__, pyrokinetics.__file__)
data_root = Path(os.path.expandvars(os.environ["GK_DATA_ROOT"]))   # the WORK tree: the GS2 kbm_free databases
tglf_root = Path(os.path.expandvars("$GYRO_DATA_OUTPUT"))   # the TGLF leaves are on scratch only
cases = ["SPR-045", "M1", "R4", "R3"]
filters = {"f2p0": 2.0, "f0p0": 0.0, "fm1p5": -1.5}
nbases = [(2, 4), (4, 8), (16, 16)]
leaf_prefix = {1: "fw", 2: "s2"}   # fw = qp5.34 SAT1 leaves, s2 = qp5.35 SAT2 leaves
tolerance_gate, tolerance_tight = 0.05, 0.01
n_cpu = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))
analysis_name = "tglf_sat1_sat2"

In [ ]:
def tie_species(pyro):
    # GyroRun's links for the kbm_free cubes (not stored in pyroscan.json): ions share the first ion's a/L_T, every a/L_n is the electron's
    species = pyro.local_species
    ions = [name for name in species.names if name != "electron"]
    for ion in ions[1:]:
        species[ion].inverse_lt = species[ions[0]].inverse_lt
    for ion in ions:
        species[ion].inverse_ln = species["electron"].inverse_ln
    species.enforce_quasineutrality("electron")


def species_numbers(pyro):
    return {name: (float(pyro.local_species[name].temp.m), float(pyro.local_species[name].dens.m), float(pyro.local_species[name].inverse_lt.m))
            for name in pyro.local_species.names}


def species_untied_and_tied(sample):
    pyro = scan_in_process.sample_pyro(sample)
    ky = float(pyro.numerics.ky.m)
    untied = species_numbers(pyro)
    tie_species(pyro)
    return untied, species_numbers(pyro), ky


def deck_species(directory):
    # the deck GS2 ran is stored in input.out.nc as input_file (the WORK tree keeps outputs only)
    text = xr.open_dataset(directory / "input.out.nc").input_file.values.tobytes().decode(errors="ignore").replace("\x00", "")
    deck = Path(tempfile.mkdtemp()) / "input.in"
    deck.write_text(text)
    return species_numbers(Pyro(gk_file=deck))


def bias_stats(gamma_tglf, gamma_gs2, n_boot=10000):
    found = gamma_tglf > 0
    ratio_log = np.log10(gamma_tglf[found] / gamma_gs2[found])
    boot = [np.median(rng.choice(ratio_log, ratio_log.size)) for _ in range(n_boot)]
    return dict(n=gamma_tglf.size, n_found=int(found.sum()), n_miss=int((~found).sum()),
                frac_above=float((ratio_log > 0).mean()), median_log_ratio=float(np.median(ratio_log)),
                ci_lo=float(np.percentile(boot, 2.5)), ci_hi=float(np.percentile(boot, 97.5)),
                mean_ratio=float((gamma_tglf[found] / gamma_gs2[found]).mean()),
                logmed_abs_err=float(np.log10(np.median(np.abs(gamma_tglf - gamma_gs2)))))


def extent_of(power_values, theta_values):
    # pyro's own 95% rule (Extent._compute, the function Extent(output) calls); power_values: (sample, theta)
    power = xr.DataArray(power_values, dims=("sample", "theta"), coords={"theta": theta_values})
    return Extent._compute(power, 0.95)[0].values

## Per case: GS2 population and extent, then the eight TGLF arms per SAT rule

One database per loop iteration (pyro's `enforce_consistent_beta_prime` is state-dependent across scans created in one process, qp5.33).
GS2 `phi_squared` sits on a union $\theta$ grid with NaN outside each sample's own points, so the extent is taken per sample on its finite points.

In [ ]:
gs2, tglf, population_rows, tie_rows, check_rows = {}, {}, [], [], []
for case in cases:
    leaf_gs2 = data_root / "GS2/Runs/LATIN_HYPERCUBE" / case / "kbm_free"
    gs2_scan = PyroHypercube(pyroscan_json=leaf_gs2 / "pyroscan.json", load_base_pyro=True)
    gs2_scan.load_gk_output(netcdf_file=leaf_gs2 / "pyroscan.nc")
    gs2_output = gs2_scan.gk_output.data
    names = np.array([str(name) for name in gs2_output.sample_name.values])
    assert list(names) == list(gs2_scan.pyro_dict)
    scan_in_process = gs2_scan

    with get_context("fork").Pool(n_cpu) as pool:
        triples = pool.map(species_untied_and_tied, range(len(names)), chunksize=8)
    untied, tied, ky = [t[0] for t in triples], [t[1] for t in triples], np.array([t[2] for t in triples])
    for sample in (0, 1, 7, len(names) // 3, len(names) // 2, len(names) - 1):
        deck = deck_species(leaf_gs2 / names[sample])
        error = max(abs(tied[sample][s][k] - deck[s][k]) / max(abs(deck[s][k]), 1e-12) for s in deck for k in range(3))
        check_rows.append(dict(case=case, sample=names[sample], max_rel_species_error_tied=error,
                               max_rel_species_error_untied=max(abs(untied[sample][s][k] - deck[s][k]) / max(abs(deck[s][k]), 1e-12) for s in deck for k in range(3))))
    assert max(r["max_rel_species_error_tied"] for r in check_rows if r["case"] == case) < 1e-6, f"{case}: tied species differ from the decks"

    heat = gs2_output.heat.sum("field").pint.dequantify()
    particle = gs2_output.particle.sum("field").pint.dequantify()
    def chi(group, species):
        numerator = denominator = 0.0
        for s in group:
            temp, dens, inverse_lt = np.array([sp[s] for sp in species]).T
            numerator = numerator + heat.sel(species=s).values - 1.5 * temp * particle.sel(species=s).values
            denominator = denominator + dens * temp * inverse_lt
        return numerator / denominator
    ions = [str(s) for s in gs2_output.species.values if s != "electron"]
    labels = {}
    with np.errstate(divide="ignore", invalid="ignore"):
        for tie_name, species in (("tied", tied), ("untied", untied)):
            labels[tie_name] = np.asarray(mc.classify(dict(
                T=gs2_output.tearing_parameter.values.ravel(),
                omega=mc.ION_DIRECTION * gs2_output.mode_frequency.pint.dequantify().values.ravel(),
                chi_ratio=chi(["electron"], species) / chi(ions, species))))
    gamma = gs2_output.growth_rate.pint.dequantify().values.ravel()
    tolerance = gs2_output.growth_rate_tolerance.pint.dequantify().values.ravel()
    omega = gs2_output.mode_frequency.pint.dequantify().values.ravel()
    positive = np.isfinite(gamma) & (gamma > 0)
    kbm = positive & (tolerance < tolerance_gate) & (labels["tied"] == "KBM")
    kbm_untied = positive & (tolerance < tolerance_gate) & (labels["untied"] == "KBM")
    population_rows.append(dict(case=case, samples=len(names), gamma_positive=int(positive.sum()),
                                kbm_tol005=int(kbm.sum()), kbm_tol001=int((positive & (tolerance < tolerance_tight) & (labels["tied"] == "KBM")).sum()),
                                kbm_untied_tol005=int(kbm_untied.sum()), label_changed_by_ties=int((labels["tied"] != labels["untied"]).sum()),
                                kbm_gained_by_ties=int((kbm & ~kbm_untied).sum()), kbm_lost_by_ties=int((~kbm & kbm_untied).sum())))
    print(population_rows[-1], flush=True)

    # GS2 extent of phi, KBM samples only
    selected_gs2 = gs2_output.set_index(sample="sample_name").sel(sample=names[kbm]).phi_squared.pint.dequantify().squeeze("kx")
    theta_gs2 = selected_gs2.theta.values
    gs2_extent = np.empty(kbm.sum())
    for row, power in enumerate(selected_gs2.values):
        finite = np.isfinite(power)
        gs2_extent[row] = extent_of(power[finite][None, :], theta_gs2[finite])[0]
    print(case, "GS2 extent: median", np.median(gs2_extent), "theta range", theta_gs2.min(), theta_gs2.max(), flush=True)
    # GS2 extent of A_par (qp5.39), same rule on the cube's apar_squared
    selected_gs2_apar = gs2_output.set_index(sample="sample_name").sel(sample=names[kbm]).apar_squared.pint.dequantify().squeeze("kx")
    gs2_extent_apar = np.empty(kbm.sum())
    for row, power in enumerate(selected_gs2_apar.values):
        finite = np.isfinite(power)
        gs2_extent_apar[row] = extent_of(power[finite][None, :], selected_gs2_apar.theta.values[finite])[0]
    print(case, "GS2 A_par extent: median", np.median(gs2_extent_apar), flush=True)
    gs2[case] = dict(names=names[kbm], gamma=gamma[kbm], omega=omega[kbm], ky=ky[kbm], extent=gs2_extent, extent_apar=gs2_extent_apar, units=str(gs2_output.growth_rate.data.units))
    del selected_gs2, selected_gs2_apar, gs2_output

    tglf[case] = {}
    for sat_rule in (1, 2):
        for filter_key in filters:
            for nb_min, nb_max in nbases:
                leaf = tglf_root / "TGLF/Runs/LATIN_HYPERCUBE" / case / "kbm_free" / f"{leaf_prefix[sat_rule]}_{filter_key}_nb{nb_min}-{nb_max}"
                tglf_scan = PyroHypercube(pyroscan_json=leaf / "pyroscan.json", load_base_pyro=True)
                tglf_scan.load_gk_output(netcdf_file=leaf / "pyroscan.nc")
                tglf_output = tglf_scan.gk_output.data
                assert [str(n) for n in tglf_output.sample_name.values] == list(names)
                selected = tglf_output.set_index(sample="sample_name").sel(sample=gs2[case]["names"])
                assert re.sub(r"_pyroscan_base\d+", "", str(selected.growth_rate.data.units)) == re.sub(r"_pyroscan_base\d+", "", gs2[case]["units"]), "normalisations differ"
                growth = selected.growth_rate.pint.dequantify()
                flags = json.loads((leaf / "manifest.json").read_text())["code_flags"]   # settings GyroRun put on the source deck
                assert flags["IBRANCH"] == -1 and flags["FILTER"] == filters[filter_key] and (flags["NBASIS_MIN"], flags["NBASIS_MAX"]) == (nb_min, nb_max)
                assert flags.get("SAT_RULE", 1) == sat_rule   # qp5.34 fw_* leaves carry no SAT_RULE flag: the pyro template's 1
                assert (growth.fillna(-np.inf).argmax("mode") == 0).all()  # mode 0 = dominant
                gamma_tglf = growth.isel(mode=0).clip(min=0).values
                phi = selected.phi.isel(mode=0).pint.dequantify().squeeze("kx").transpose("sample", "theta")   # kx has one point
                theta_tglf = np.asarray(phi.theta.values, dtype=float)
                found = gamma_tglf > 0
                power = np.abs(phi.values) ** 2
                assert np.isfinite(power[found]).all(), "NaN in a TGLF phi with gamma>0"
                extent_tglf = np.full(len(gamma_tglf), np.nan)
                extent_tglf[found] = extent_of(power[found], theta_tglf)
                apar = selected.apar.isel(mode=0).pint.dequantify().squeeze("kx").transpose("sample", "theta")
                power_apar = np.abs(apar.values) ** 2
                assert np.isfinite(power_apar[found]).all(), "NaN in a TGLF apar with gamma>0"
                extent_tglf_apar = np.full(len(gamma_tglf), np.nan)
                extent_tglf_apar[found] = extent_of(power_apar[found], theta_tglf)
                tglf[case][sat_rule, filter_key, nb_min, nb_max] = dict(
                    gamma=gamma_tglf,
                    omega=selected.mode_frequency.pint.dequantify().isel(mode=0).values,
                    width=selected.gaussian_width.pint.dequantify().values,
                    extent=extent_tglf, extent_apar=extent_tglf_apar)
                del tglf_output, selected, phi, power, apar, power_apar
    print(case, "TGLF arms loaded:", len(tglf[case]), "TGLF theta range", theta_tglf.min(), theta_tglf.max(), flush=True)
population = pd.DataFrame(population_rows)
print(population.to_string())
print(pd.DataFrame(check_rows).to_string())

## Bias table: case $\times$ SAT rule $\times$ FILTER $\times$ NBASIS, plus a pooled row per setting

The bootstrap restarts from seed 0 for each SAT rule, so the SAT1 rows are bit-identical to qp5.34 (`results/tglf_filter_width/csv/bias_by_case_setting.csv`).

In [ ]:
def pooled_arm(sat_rule, key):
    arms = [tglf[case][(sat_rule, *key)] for case in cases]
    return {name: np.concatenate([arm[name] for arm in arms]) for name in ("gamma", "omega", "width", "extent", "extent_apar")}

pooled_gs2 = {name: np.concatenate([gs2[case][name] for case in cases]) for name in ("gamma", "omega", "ky", "extent", "extent_apar")}
pooled_case = np.concatenate([[case] * len(gs2[case]["gamma"]) for case in cases])
setting_keys = [(filter_key, nb_min, nb_max) for filter_key in filters for nb_min, nb_max in nbases]
rows = []
for sat_rule in (1, 2):
    rng = np.random.default_rng(0)
    for case in cases:
        for key in setting_keys:
            arm = tglf[case][(sat_rule, *key)]
            rows.append(dict(sat_rule=sat_rule, case=case, filter=filters[key[0]], nbasis_min=key[1], nbasis_max=key[2], **bias_stats(arm["gamma"], gs2[case]["gamma"])))
    for key in setting_keys:
        rows.append(dict(sat_rule=sat_rule, case="pooled", filter=filters[key[0]], nbasis_min=key[1], nbasis_max=key[2],
                         **bias_stats(pooled_arm(sat_rule, key)["gamma"], pooled_gs2["gamma"])))
summary = pd.DataFrame(rows)
print(summary.round(3).to_string())

## Extent ratio $\theta_{\rm TGLF}/\theta_{\rm GS2}$ per case/pooled $\times$ SAT $\times$ setting

Medians over TGLF-found samples. "bias below/above" = median $\log_{10}(\gamma_{\rm TGLF}/\gamma_{\rm GS2})$ in the half with ratio $<1$ / $>1$. Spearman is $\theta_{\rm TGLF}$ against the selected `gaussian_width` (same samples).

In [ ]:
def extent_stats(arm, reference, field="extent"):
    found = (arm["gamma"] > 0) & np.isfinite(arm[field])
    ratio = arm[field][found] / reference[field][found]
    bias = np.log10(arm["gamma"][found] / reference["gamma"][found])
    return dict(n=int(found.sum()), median_extent_ratio=float(np.median(ratio)), frac_ratio_lt1=float((ratio < 1).mean()), frac_ratio_gt1=float((ratio > 1).mean()),
                median_bias_ratio_lt1=float(np.median(bias[ratio < 1])) if (ratio < 1).any() else np.nan,
                median_bias_ratio_gt1=float(np.median(bias[ratio > 1])) if (ratio > 1).any() else np.nan,
                median_extent_tglf=float(np.median(arm[field][found])), median_extent_gs2=float(np.median(reference[field][found])),
                spearman_extent_width=float(spearmanr(arm[field][found], arm["width"][found]).statistic))

extent_rows = []
for sat_rule in (1, 2):
    for key in setting_keys:
        for case in cases:
            extent_rows.append(dict(sat_rule=sat_rule, case=case, filter=filters[key[0]], nbasis_min=key[1], nbasis_max=key[2],
                                    **extent_stats(tglf[case][(sat_rule, *key)], gs2[case])))
        extent_rows.append(dict(sat_rule=sat_rule, case="pooled", filter=filters[key[0]], nbasis_min=key[1], nbasis_max=key[2],
                                **extent_stats(pooled_arm(sat_rule, key), pooled_gs2)))
extent_table = pd.DataFrame(extent_rows)
print(extent_table.round(3).to_string())
# A_par analogue (qp5.39): same rows and columns
apar_rows = []
for sat_rule in (1, 2):
    for key in setting_keys:
        for case in cases:
            apar_rows.append(dict(sat_rule=sat_rule, case=case, filter=filters[key[0]], nbasis_min=key[1], nbasis_max=key[2],
                                  **extent_stats(tglf[case][(sat_rule, *key)], gs2[case], "extent_apar")))
        apar_rows.append(dict(sat_rule=sat_rule, case="pooled", filter=filters[key[0]], nbasis_min=key[1], nbasis_max=key[2],
                              **extent_stats(pooled_arm(sat_rule, key), pooled_gs2, "extent_apar")))
apar_extent_table = pd.DataFrame(apar_rows)
print(apar_extent_table.round(3).to_string())

## Figures: 6 kinds $\times$ (4 cases + pooled) $\times$ SAT1/SAT2 = 60

3$\times$3 grid (rows FILTER 2 / 0 / $-1.5$, columns NBASIS (2,4) / (4,8) / (16,16)), format of qp5.30 / qp5.34 fig1. Only samples with TGLF $\gamma>0$ are drawn.

In [ ]:
Path("plots").mkdir(exist_ok=True)
case_colour = {"SPR-045": "tab:blue", "M1": "tab:orange", "R4": "tab:green", "R3": "tab:red"}
kinds = {"gamma": "gamma_vs_gs2", "gamma_ky": "gamma_vs_gs2_colour_ky", "gamma_ratio": "gamma_vs_gs2_colour_extent_ratio",
         "gamma_split": "gamma_vs_gs2_colour_extent_ratio_split", "extent_gamma": "extent_tglf_vs_gs2_colour_gamma", "extent_ky": "extent_tglf_vs_gs2_colour_ky"}

def figure(sat_rule, group, kind, field="extent"):
    members = cases if group == "pooled" else [group]
    fig, axes = plt.subplots(3, 3, figsize=(15, 15), sharex=True, sharey=True, constrained_layout=True)
    mappable = None
    for ax, key in zip(axes.ravel(), setting_keys):
        arm = pooled_arm(sat_rule, key) if group == "pooled" else tglf[group][(sat_rule, *key)]
        reference = pooled_gs2 if group == "pooled" else gs2[group]
        found = (arm["gamma"] > 0) & np.isfinite(arm[field])
        ratio = arm[field] / reference[field]
        stats = summary[(summary.sat_rule == sat_rule) & (summary.case == group) & (summary["filter"] == filters[key[0]]) & (summary.nbasis_min == key[1])].iloc[0]
        if kind.startswith("extent"):
            x, y = reference[field][found], arm[field][found]
            colour_values = arm["gamma"][found] if kind == "extent_gamma" else reference["ky"][found]
            mappable = ax.scatter(x, y, s=18, alpha=0.8, c=colour_values, norm=LogNorm(), cmap="viridis")
            ax.set_xscale("log"); ax.set_yscale("log")
            ax.text(0.97, 0.03, f"n={found.sum()}\nmedian ratio {np.median(y / x):.2f}", transform=ax.transAxes, ha="right", va="bottom", fontsize=10, bbox=dict(fc="white", alpha=0.85))
        else:
            x, y = reference["gamma"][found], arm["gamma"][found]
            if kind == "gamma" and group == "pooled":
                for case in cases:
                    mask = pooled_case[found] == case
                    ax.scatter(x[mask], y[mask], s=14, alpha=0.7, color=case_colour[case], label=f"{case} (n={mask.sum()})")
            elif kind == "gamma":
                ax.scatter(x, y, s=18, alpha=0.7, color=case_colour[group], label=f"{group} (n={found.sum()})")
            elif kind == "gamma_ky":
                mappable = ax.scatter(x, y, s=18, alpha=0.8, c=reference["ky"][found], norm=LogNorm(), cmap="viridis")
            elif kind == "gamma_ratio":
                mappable = ax.scatter(x, y, s=18, alpha=0.8, c=np.log10(ratio[found]), vmin=-1, vmax=1, cmap="coolwarm")
            else:  # gamma_split
                below = ratio[found] < 1
                ax.scatter(x[below], y[below], s=18, alpha=0.7, color="tab:blue", label=rf"$\theta_T/\theta_G<1$ (n={below.sum()})")
                ax.scatter(x[~below], y[~below], s=18, alpha=0.7, color="tab:red", label=rf"$\theta_T/\theta_G>1$ (n={(~below).sum()})")
            ax.set_xscale("log"); ax.set_yscale("log")
            ax.text(0.97, 0.03, f"n={stats.n_found} (+{stats.n_miss} TGLF misses)\nabove $y=x$: {100*stats.frac_above:.0f}%\n"
                    f"median $\\log_{{10}}(\\gamma_T/\\gamma_G)$ = {stats.median_log_ratio:+.2f} [{stats.ci_lo:+.2f}, {stats.ci_hi:+.2f}]\n"
                    f"mean ratio {stats.mean_ratio:.2f}\nLogMed $|\\gamma$ err$|$ = {stats.logmed_abs_err:.2f}",
                    transform=ax.transAxes, ha="right", va="bottom", fontsize=10, bbox=dict(fc="white", alpha=0.85))
        ax.set_title(f"SAT{sat_rule}, FILTER {filters[key[0]]:g}, NBASIS {key[1]}-{key[2]}", fontsize=11)
    quantity = (r"\theta^{A_\parallel}" if field == "extent_apar" else r"\theta") if kind.startswith("extent") else r"\gamma"
    for ax in axes[:, 0]: ax.set_ylabel(rf"${quantity}_{{\rm TGLF}}$")
    for ax in axes[-1]: ax.set_xlabel(rf"${quantity}_{{\rm GS2}}$")
    low = min(min(ax.get_xlim()[0], ax.get_ylim()[0]) for ax in axes.ravel()); high = max(max(ax.get_xlim()[1], ax.get_ylim()[1]) for ax in axes.ravel())
    for ax in axes.ravel():
        ax.plot([low, high], [low, high], "k--", lw=1); ax.set_xlim(low, high); ax.set_ylim(low, high); ax.set_aspect("equal")
    if mappable is not None:
        label = {"gamma_ky": r"$k_y$ (pyro units)", "extent_ky": r"$k_y$ (pyro units)", "extent_gamma": r"$\gamma_{\rm TGLF}$",
                 "gamma_ratio": r"$\log_{10}(\theta_{\rm TGLF}/\theta_{\rm GS2})$"}[kind]
        fig.colorbar(mappable, ax=axes, shrink=0.6, label=label)
    if kind in ("gamma", "gamma_split"):
        axes[0, 0].legend(loc="upper left")
    path = f"plots/fig_sat{sat_rule}_{'apar_' if field == 'extent_apar' else ''}{kinds[kind]}_{group}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)
    return path

made = []
for sat_rule in (1, 2):
    for group in (*cases, "pooled"):
        for kind in kinds:
            made.append(figure(sat_rule, group, kind))
        print("figures done", sat_rule, group, flush=True)
assert len(made) == 60
made_apar = []   # qp5.39: A_par versions of the two extent-vs-extent kinds, 2 SAT x 5 groups x 2 kinds = 20
for sat_rule in (1, 2):
    for group in (*cases, "pooled"):
        for kind in ("extent_gamma", "extent_ky"):
            made_apar.append(figure(sat_rule, group, kind, "extent_apar"))
assert len(made_apar) == 20

## SAT2 vs SAT1 selected width (same samples, same FILTER and NBASIS; TGLF misses dropped from a pair)

Format of the qp5.34 FILTER-pair table with `sat_a` = 2, `sat_b` = 1. Mode changed = sign of $\omega$ flips or $\lvert\omega\rvert$ moves by a factor 2.

In [ ]:
width_rows = []
for case in cases:
    for key in setting_keys:
        arm_a, arm_b = tglf[case][(2, *key)], tglf[case][(1, *key)]
        both = (arm_a["gamma"] > 0) & (arm_b["gamma"] > 0)
        ratio = arm_a["width"][both] / arm_b["width"][both]
        omega_a, omega_b = arm_a["omega"][both], arm_b["omega"][both]
        mode_changed = (np.sign(omega_a) != np.sign(omega_b)) | (np.abs(omega_a) > 2 * np.abs(omega_b)) | (np.abs(omega_b) > 2 * np.abs(omega_a))
        width_b = arm_b["width"][both]
        width_rows.append(dict(
            case=case, nbasis_min=key[1], nbasis_max=key[2], filter=filters[key[0]], sat_a=2, sat_b=1, n_pairs=int(both.sum()),
            frac_width_change_gt5pct=float((np.abs(ratio - 1) > 0.05).mean()), frac_width_change_gt50pct=float((np.abs(ratio - 1) > 0.5).mean()),
            median_width_ratio=float(np.median(ratio)), median_width_sat1=float(np.median(width_b)), median_width_sat2=float(np.median(arm_a["width"][both])),
            frac_mode_changed=float(mode_changed.mean()),
            median_abs_omega_sat1_changed=float(np.median(np.abs(omega_b[mode_changed]))) if mode_changed.any() else np.nan,
            median_abs_omega_sat1_unchanged=float(np.median(np.abs(omega_b[~mode_changed])))))
width_table = pd.DataFrame(width_rows)
print(width_table.round(3).to_string())

In [ ]:
Path("csv").mkdir(exist_ok=True)
summary.to_csv("csv/bias_by_case_setting.csv", index=False)
extent_table.to_csv("csv/extent_ratio_by_case_setting.csv", index=False)
apar_extent_table.to_csv("csv/apar_extent_ratio_by_case_setting.csv", index=False)
width_table.to_csv("csv/width_change_sat2_vs_sat1.csv", index=False)
population.to_csv("csv/kbm_population.csv", index=False)
pd.DataFrame(check_rows).to_csv("csv/species_tie_deck_check.csv", index=False)
print(sorted(p.name for p in Path(".").rglob("*") if p.is_file()))